# 把你的数据叠加到论文图片上进行比对

这个 notebook 的思路是：
1. 读入你师姐论文里的图片；
2. 手动给出图中坐标轴在图片里的像素范围；
3. 读入你自己的数据；
4. 把你的数据按坐标范围映射到图片上；
5. 叠加显示并导出结果。

适合你现在这种：手头只有一张处理过的图，没有原始绘图脚本。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams['figure.dpi'] = 140
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 读入背景图片

把你的图片路径填在 `img_path`。当前 notebook 默认用你上传的这张图。

In [ ]:
img_path = '/mnt/data/test1.png'   # 你的论文图
img = np.array(Image.open(img_path))

h, w = img.shape[:2]
print(f'image size = {w} x {h}')

plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.title('原图')
plt.axis('off')
plt.show()

## 2. 先看像素坐标，确定绘图区边界

这一步是关键。

你需要找到**真正的坐标轴矩形框**在图片中的像素范围：
- `x_left`：绘图区左边界
- `x_right`：绘图区右边界
- `y_top`：绘图区上边界
- `y_bottom`：绘图区下边界

下面这个单元会显示图片，并允许你点击查看大概坐标。

### 推荐做法
直接点四个角附近，记下坐标后填到下一格。

按照你这张图，我已经给了一组**大致可用的初值**，但你最好自己再微调一下。

In [ ]:
%matplotlib notebook
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img)
ax.set_title('点击图片查看像素坐标；看完后再运行后面的格子')
plt.show()

如果你不想点坐标，也可以先用下面这组近似值试一下：

- 左边界约 `x_left = 255`
- 右边界约 `x_right = 1438`
- 上边界约 `y_top = 80`
- 下边界约 `y_bottom = 810`

这是按你这张图肉眼估的，通常够你先跑通。

In [ ]:
# ===== 手动填写绘图区像素边界 =====
# 建议你根据点击结果微调
x_left   = 255
x_right  = 1438
y_top    = 80
y_bottom = 810

# ===== 图中真实坐标范围 =====
x_min, x_max = 20, 180          # theta (deg)
y_min, y_max = -1, 3            # Log10(|dσ/dΩ|)

print('plot box pixels:')
print(x_left, x_right, y_top, y_bottom)
print('data range:')
print((x_min, x_max), (y_min, y_max))

## 3. 读入你自己的数据

最方便的格式是一个 CSV 文件，两列：
- 第一列：`theta`
- 第二列：`y`

比如：

```csv
theta,y
20,2.95
25,2.70
30,2.50
...
```

如果你的列名不一样，也没关系，下面可以改。

In [ ]:
# ===== 把这里改成你的数据文件路径 =====
csv_path = 'my_data.csv'

# 例：如果你还没准备 csv，可以先取消下面这段注释，生成一个示例文件
# demo_theta = np.linspace(22, 177, 220)
# demo_y = 2.8*np.exp(-(demo_theta-20)/55) + 0.25*np.sin(demo_theta/1.7) + 0.2
# pd.DataFrame({'theta': demo_theta, 'y': demo_y}).to_csv(csv_path, index=False)
# print(f'已生成示例文件: {csv_path}')

df = pd.read_csv(csv_path)
display(df.head())

In [ ]:
# ===== 这里指定你的列名 =====
x_col = 'theta'
y_col = 'y'

x = df[x_col].to_numpy()
y = df[y_col].to_numpy()

print('x range:', x.min(), '->', x.max())
print('y range:', y.min(), '->', y.max())

## 4. 把数据映射到图片像素坐标

注意：图片坐标的 y 轴是**向下增大**，而数学坐标是**向上增大**，所以 y 要反过来映射。

In [ ]:
def data_to_pixel_x(x, x_min, x_max, x_left, x_right):
    return x_left + (x - x_min) / (x_max - x_min) * (x_right - x_left)

def data_to_pixel_y(y, y_min, y_max, y_top, y_bottom):
    return y_bottom - (y - y_min) / (y_max - y_min) * (y_bottom - y_top)

xp = data_to_pixel_x(x, x_min, x_max, x_left, x_right)
yp = data_to_pixel_y(y, y_min, y_max, y_top, y_bottom)

## 5. 叠加显示

这里我给你两种画法：
- 只画散点；
- 画连线+散点。

通常跟论文曲线比对时，`line + marker` 更直观。

In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img)

# 你的数据叠加到图片上
ax.plot(xp, yp, '-', lw=2.0, label='My data')
ax.plot(xp, yp, 'o', ms=4, label='My points')

# 可选：把绘图区框出来，方便检查映射是否准
rect_x = [x_left, x_right, x_right, x_left, x_left]
rect_y = [y_top, y_top, y_bottom, y_bottom, y_top]
ax.plot(rect_x, rect_y, '--', lw=1.2, label='Plot box')

ax.set_title('Overlay comparison')
ax.set_xlim(0, w)
ax.set_ylim(h, 0)
ax.legend()
plt.show()

## 6. 更推荐：按真实坐标重新画一张“干净版对比图”

如果你手头已经有自己的数据，而且师姐论文曲线你也能大概数字化出来，
其实最规范的是不要直接压在截图上，而是在同一坐标系重画。

但如果你现在只是为了快速肉眼比对，那么截图叠加已经很好用了。

In [ ]:
# 在真实坐标里展示你的数据，便于检查
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, y, '-o', ms=3, label='My data')
ax.set_xlabel(r'$\theta$ (deg)')
ax.set_ylabel(r'Log$_{10}(|\mathrm{d}\sigma/\mathrm{d}\Omega|)$')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 7. 导出叠加结果

In [ ]:
out_path = 'overlay_compare.png'

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img)
ax.plot(xp, yp, '-', lw=2.0, label='My data')
ax.plot(xp, yp, 'o', ms=4, label='My points')
ax.set_xlim(0, w)
ax.set_ylim(h, 0)
ax.legend()
ax.set_title('Overlay comparison')

plt.tight_layout()
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'已保存: {out_path}')

## 8. 你最可能需要改的地方

### 情况 A：叠加位置整体偏一点
优先微调这四个值：
- `x_left`
- `x_right`
- `y_top`
- `y_bottom`

### 情况 B：左右对得上，但上下不对
检查：
- `y_min, y_max` 是否正确；
- 你的 y 数据是不是已经取过 `log10`；
- 图上的纵轴是不是和你数据单位完全一致。

### 情况 C：你的数据不是 CSV，而是两个 numpy 数组
那你可以直接跳过 `read_csv`，改成：

```python
x = np.array([...])
y = np.array([...])
```

### 情况 D：你想画成红线
把：
```python
ax.plot(xp, yp, '-', lw=2.0)
```
改成：
```python
ax.plot(xp, yp, 'r-', lw=2.0)
```

不过为了和原图里黑线区分，一般确实建议你用红色或蓝色。